# EDA Cartier QTEM — Fase 5+6: Quality Analysis e Referential Integrity

**Obiettivo**: identificare problemi di qualità per ogni dataset e verificare la consistenza referenziale tra dataset.  
**Read-only**: nessuna modifica ai dati originali.  
**Output**: tabelle CSV in `output/tables/`, finding stampati con header.


In [ ]:
import sys
from pathlib import Path

# Aggiungi la cartella scripts al path
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / 'scripts'))

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from utils import (
    load_all_datasets,
    print_finding,
    OUT_TABLES,
    OUT_PLOTS,
)

# Caricamento tutti i dataset
dfs = load_all_datasets()
agg  = dfs['Aggregated_Data']
trs  = dfs['Transactions']
cli  = dfs['Clients']
crc  = dfs['CRC']
ccp  = dfs['CCP']
art  = dfs['Articles']

print(f"Aggregated_Data: {agg.shape}")
print(f"Transactions:    {trs.shape}")
print(f"Clients:         {cli.shape}")
print(f"CRC:             {crc.shape}")
print(f"CCP:             {ccp.shape}")
print(f"Articles:        {art.shape}")

# Snapshot 2021
agg_2021 = agg[agg['DATE_TARGET'].dt.year == 2021].copy()
print(f"\nAggregated_Data snapshot 2021: {agg_2021.shape}")
clients_2021 = set(agg_2021['CLIENT_ID'].dropna().unique())
print(f"CLIENT_ID unici in snapshot 2021: {len(clients_2021):,}")

---
## Fase 5A — Transactions: copertura temporale e leakage risk

In [ ]:
# 5A.1 — Intervallo reale TRS_DATE
print("=== 5A — TRANSACTIONS: COPERTURA TEMPORALE ===")
print(f"TRS_DATE min: {trs['TRS_DATE'].min()}")
print(f"TRS_DATE max: {trs['TRS_DATE'].max()}")
print(f"NULL TRS_DATE: {trs['TRS_DATE'].isna().sum():,}")

# 5A.2 — Per ogni snapshot DATE_TARGET: safe vs leakage
DATE_TARGETS = sorted(agg['DATE_TARGET'].dropna().unique())
print(f"\nSnapshot DATE_TARGET: {[str(d)[:10] for d in DATE_TARGETS]}")

rows_cov = []
for snap in DATE_TARGETS:
    safe  = (trs['TRS_DATE'] <= snap).sum()
    leak  = (trs['TRS_DATE'] >  snap).sum()
    total = safe + leak
    rows_cov.append({
        'DATE_TARGET':    str(snap)[:10],
        'TRS safe (<= DT)': int(safe),
        'TRS leakage (> DT)': int(leak),
        'Total':          int(total),
        '% leakage':      round(leak / total * 100, 2) if total > 0 else 0,
    })

cov_df = pd.DataFrame(rows_cov)
print("\nCopertura temporale per snapshot:")
print(cov_df.to_string(index=False))

# 5A.3 — Distribuzione leakage per CHANNEL
# Prende transazioni con TRS_DATE > 2021 (ultimo snapshot) come caso più critico
snap_last = DATE_TARGETS[-1]
trs_leak_last = trs[trs['TRS_DATE'] > snap_last]
print(f"\nTransazioni con TRS_DATE > {str(snap_last)[:10]}: {len(trs_leak_last):,}")
if len(trs_leak_last) > 0:
    print("\nDistribuzione per CHANNEL (post-ultimo snapshot):")
    print(trs_leak_last['CHANNEL'].value_counts().to_string())
    print("\nDistribuzione per TRS_CATEG:")
    print(trs_leak_last['TRS_CATEG'].value_counts().to_string())
    print("\nDistribuzione per anno:")
    print(trs_leak_last['TRS_DATE'].dt.year.value_counts().sort_index().to_string())

print_finding(
    'TRANSACTIONS COPERTURA TEMPORALE',
    f"TRS_DATE va da {trs['TRS_DATE'].min()} a {trs['TRS_DATE'].max()}.\n"
    f"Transazioni post snapshot-2021: {len(trs_leak_last):,} ({len(trs_leak_last)/len(trs)*100:.1f}% del totale).\n"
    "Queste non devono entrare come feature nei modelli basati sullo snapshot 2021.\n"
    "Tabella completa salvata in output/tables/."
)

cov_df.to_csv(OUT_TABLES / 'transactions_temporal_coverage.csv', index=False)
print("\nSalvato: transactions_temporal_coverage.csv")

---
## Fase 5B — Transactions: SERIAL_NUMBER placeholder

In [ ]:
# 5B.1 — Conteggi NULL e placeholder
PLACEHOLDER = trs['SERIAL_NUMBER'].value_counts().idxmax() if trs['SERIAL_NUMBER'].notna().any() else None
n_null = int(trs['SERIAL_NUMBER'].isna().sum())
n_ph   = int((trs['SERIAL_NUMBER'] == PLACEHOLDER).sum()) if PLACEHOLDER else 0

print(f"=== 5B — SERIAL_NUMBER ===")
print(f"NULL SERIAL_NUMBER:    {n_null:,} ({n_null/len(trs)*100:.1f}%)")
print(f"Placeholder top value: {PLACEHOLDER!r}")
print(f"N. occorrenze placeholder: {n_ph:,} ({n_ph/len(trs)*100:.1f}%)")
print(f"TOP 10 SERIAL_NUMBER:")
print(trs['SERIAL_NUMBER'].value_counts().head(10).to_string())

def serial_breakdown(mask: pd.Series, label: str) -> pd.DataFrame:
    """Analisi distribuzione per TRS_CATEG, CHANNEL, CATEG, anno."""
    sub = trs[mask]
    print(f"\n--- {label} ({len(sub):,} righe) ---")
    for col in ['TRS_CATEG', 'CHANNEL', 'CATEG']:
        print(f"\n  {col}:")
        vc = sub[col].value_counts(dropna=False)
        print(vc.to_string())
    print("\n  Anno TRS_DATE:")
    print(sub['TRS_DATE'].dt.year.value_counts().sort_index().to_string())
    return sub

# Analisi NULL
sub_null = serial_breakdown(trs['SERIAL_NUMBER'].isna(), 'SERIAL_NUMBER NULL')

# Analisi placeholder
if PLACEHOLDER:
    sub_ph = serial_breakdown(trs['SERIAL_NUMBER'] == PLACEHOLDER, f'SERIAL_NUMBER placeholder {PLACEHOLDER[:8]}...')

# 5B.4 — Sovrapposizione sui segmenti
print("\n=== 5B.4 — Sovrapposizione NULL vs Placeholder ===")
if PLACEHOLDER:
    null_channels = set(sub_null['CHANNEL'].dropna().unique())
    ph_channels   = set(sub_ph['CHANNEL'].dropna().unique())
    print(f"CHANNEL NULL:        {sorted(null_channels)}")
    print(f"CHANNEL Placeholder: {sorted(ph_channels)}")
    print(f"Overlap CHANNEL:     {sorted(null_channels & ph_channels)}")

# Salva tabella riassuntiva
sn_rows = []
for categ, label in [('TRS_CATEG', 'TRS_CATEG'), ('CHANNEL', 'CHANNEL'), ('CATEG', 'CATEG')]:
    for val in trs[categ].dropna().unique():
        mask_val = trs[categ] == val
        total_val = mask_val.sum()
        null_val  = (mask_val & trs['SERIAL_NUMBER'].isna()).sum()
        ph_val    = (mask_val & (trs['SERIAL_NUMBER'] == PLACEHOLDER)).sum() if PLACEHOLDER else 0
        sn_rows.append({
            'Dimensione': categ, 'Valore': val,
            'N totale': total_val,
            'N NULL SN': null_val, '% NULL SN': round(null_val/total_val*100,1) if total_val else 0,
            'N Placeholder SN': ph_val, '% Placeholder SN': round(ph_val/total_val*100,1) if total_val else 0,
        })
sn_df = pd.DataFrame(sn_rows)
sn_df.to_csv(OUT_TABLES / 'serial_number_analysis.csv', index=False)

print_finding(
    'SERIAL_NUMBER',
    f"NULL: {n_null:,} ({n_null/len(trs)*100:.1f}%).\n"
    f"Placeholder '{PLACEHOLDER}': {n_ph:,} ({n_ph/len(trs)*100:.1f}%).\n"
    "Dettaglio per CHANNEL/CATEG/anno salvato in serial_number_analysis.csv."
)

---
## Fase 5C — Transactions: valori negativi TO_WITHOUTTAX_EUR_CONST

In [ ]:
col_spend = 'TO_WITHOUTTAX_EUR_CONST'

# 5C.1 — Conteggio e distribuzione valori negativi
neg_mask = trs[col_spend] < 0
null_mask = trs[col_spend].isna()
neg_trs = trs[neg_mask]

print(f"=== 5C — VALORI NEGATIVI {col_spend} ===")
print(f"NULL:             {null_mask.sum():,} ({null_mask.mean()*100:.2f}%)")
print(f"Valori < 0:       {neg_mask.sum():,} ({neg_mask.mean()*100:.3f}%)")
print(f"Valori = 0:       {(trs[col_spend] == 0).sum():,}")
print(f"Valori > 0:       {(trs[col_spend] > 0).sum():,}")

print("\nDistribuzione valori negativi:")
neg_vals = neg_trs[col_spend]
print(neg_vals.describe().to_string())
print(f"p25: {np.percentile(neg_vals, 25):.2f}")
print(f"p75: {np.percentile(neg_vals, 75):.2f}")

# 5C.2 — TRS_CATEG dei valori negativi
print("\nTRS_CATEG dei valori negativi:")
print(neg_trs['TRS_CATEG'].value_counts(dropna=False).to_string())

# 5C.3 — Per ogni cliente con almeno 1 negativo, verifica se ha una positiva corrispondente (stesso ARTICLE_ID)
clients_with_neg = neg_trs['CLIENT_ID'].unique()
print(f"\nClienti con almeno 1 transazione negativa: {len(clients_with_neg):,}")

# Cerca matching articolo per un campione di clienti
sample_clients = clients_with_neg[:500]
trs_sample = trs[trs['CLIENT_ID'].isin(sample_clients)]

matched = 0
unmatched = 0
for client in sample_clients:
    client_trs = trs_sample[trs_sample['CLIENT_ID'] == client]
    neg_arts   = set(client_trs[client_trs[col_spend] < 0]['ARTICLE_ID'].dropna())
    pos_arts   = set(client_trs[client_trs[col_spend] > 0]['ARTICLE_ID'].dropna())
    if neg_arts & pos_arts:
        matched += 1
    else:
        unmatched += 1

print(f"\nCampione {len(sample_clients)} clienti con negativi:")
print(f"  Con match ARTICLE_ID positivo:  {matched} ({matched/len(sample_clients)*100:.1f}%)")
print(f"  Senza match ARTICLE_ID positivo: {unmatched} ({unmatched/len(sample_clients)*100:.1f}%)")

# 5C.4 — Concentrazione temporale, per canale e categoria
print("\nDistribuzione per CHANNEL:")
print(neg_trs['CHANNEL'].value_counts(dropna=False).to_string())
print("\nDistribuzione per CATEG:")
print(neg_trs['CATEG'].value_counts(dropna=False).to_string())
print("\nDistribuzione per anno TRS_DATE:")
print(neg_trs['TRS_DATE'].dt.year.value_counts().sort_index().to_string())

# Salva tabella riassuntiva
neg_summary = pd.DataFrame([{
    'N valori < 0': int(neg_mask.sum()),
    '% su totale': round(neg_mask.mean()*100, 3),
    'N NULL': int(null_mask.sum()),
    'min': round(neg_vals.min(), 2),
    'p25': round(np.percentile(neg_vals, 25), 2),
    'p50': round(np.percentile(neg_vals, 50), 2),
    'p75': round(np.percentile(neg_vals, 75), 2),
    'max': round(neg_vals.max(), 2),
    'N clienti con neg': len(clients_with_neg),
    'TRS_CATEG Sale': int((neg_trs['TRS_CATEG'] == 'Sale').sum()),
    'TRS_CATEG Repair': int((neg_trs['TRS_CATEG'] == 'Repair').sum()),
}])

neg_channel = neg_trs['CHANNEL'].value_counts(dropna=False).reset_index()
neg_channel.columns = ['CHANNEL', 'N negativi']

qt_rows = [neg_summary]
# Salva in quality_transactions.csv
neg_summary.to_csv(OUT_TABLES / 'quality_transactions_negatives.csv', index=False)

print_finding(
    'VALORI NEGATIVI TRANSACTIONS',
    f"Righe con {col_spend} < 0: {neg_mask.sum():,} ({neg_mask.mean()*100:.3f}%).\n"
    f"TRS_CATEG: {neg_trs['TRS_CATEG'].value_counts().to_dict()}.\n"
    f"Clienti con negativi e ARTICLE_ID positivo matching: {matched}/{len(sample_clients)} ({matched/len(sample_clients)*100:.0f}% campione).\n"
    "Ipotesi: valori negativi codificano resi (stessa categoria 'Sale').\n"
    "Salvato: quality_transactions_negatives.csv"
)

---
## Fase 5D — Transactions: ARTICLE_WWPRICE a zero

In [ ]:
col_price = 'ARTICLE_WWPRICE'

zero_price_mask = trs[col_price] == 0
zero_price_trs = trs[zero_price_mask]

print(f"=== 5D — {col_price} A ZERO ===")
print(f"Righe con {col_price} == 0: {zero_price_mask.sum():,} ({zero_price_mask.mean()*100:.2f}%)")
print(f"NULL {col_price}: {trs[col_price].isna().sum():,}")

print("\nTRS_CATEG per WWPRICE = 0:")
print(zero_price_trs['TRS_CATEG'].value_counts(dropna=False).to_string())

print("\nCHANNEL per WWPRICE = 0:")
print(zero_price_trs['CHANNEL'].value_counts(dropna=False).to_string())

print("\nCAT per WWPRICE = 0:")
print(zero_price_trs['CATEG'].value_counts(dropna=False).to_string())

# 5D.3 — TO_WITHOUTTAX_EUR_CONST per WWPRICE = 0
print("\nDistribuzione TO_WITHOUTTAX_EUR_CONST quando WWPRICE = 0:")
print(zero_price_trs[col_spend].describe().to_string())
print(f"TO_WITHOUTTAX = 0 quando WWPRICE = 0: {(zero_price_trs[col_spend] == 0).sum():,}")
print(f"TO_WITHOUTTAX > 0 quando WWPRICE = 0: {(zero_price_trs[col_spend] > 0).sum():,}")
print(f"TO_WITHOUTTAX < 0 quando WWPRICE = 0: {(zero_price_trs[col_spend] < 0).sum():,}")

# Anno di acquisto
print("\nAnno TRS_DATE per WWPRICE = 0:")
print(zero_price_trs['TRS_DATE'].dt.year.value_counts().sort_index().to_string())

print_finding(
    'WWPRICE ZERO',
    f"Righe con ARTICLE_WWPRICE = 0: {zero_price_mask.sum():,} ({zero_price_mask.mean()*100:.2f}%).\n"
    f"TRS_CATEG: {zero_price_trs['TRS_CATEG'].value_counts().to_dict()}.\n"
    "Ipotesi: repairs, gift interni o articoli con prezzo non catalogato.\n"
    f"Nota: TO_WITHOUTTAX può avere valore anche quando WWPRICE = 0."
)

---
## Fase 5E — Articles: mismatch categorie con Transactions

In [ ]:
print("=== 5E — CATEGORIE ARTICLES MANCANTI IN TRANSACTIONS ===")

cats_articles = set(art['PRODUCT_CATEGORY'].dropna().unique())
cats_trs      = set(trs['CATEG'].dropna().unique())
cats_missing  = cats_articles - cats_trs
cats_present  = cats_articles & cats_trs

print(f"Categorie in Articles:      {sorted(cats_articles)}")
print(f"Categorie in Transactions:  {sorted(cats_trs)}")
print(f"Categorie mancanti in TRS:  {sorted(cats_missing)}")
print(f"Categorie presenti in TRS:  {sorted(cats_present)}")

# 5E.1 — N. articoli unici per categoria mancante
print("\nArticoli per categoria mancante:")
art_miss = art[art['PRODUCT_CATEGORY'].isin(cats_missing)]
print(art_miss['PRODUCT_CATEGORY'].value_counts().to_string())
print(f"Totale articoli in categorie mancanti: {len(art_miss):,}")

# 5E.2 — Distribuzione WORLD_PRICE per categoria presente vs mancante
print("\nWORLD_PRICE — categorie presenti in TRS:")
print(art[art['PRODUCT_CATEGORY'].isin(cats_present)]['WORLD_PRICE'].describe().to_string())
print("\nWORLD_PRICE — categorie mancanti da TRS:")
print(art_miss['WORLD_PRICE'].describe().to_string())

# 5E.3 — FLAG_HE, FLAG_BRIDAL, FLAG_DIAMOND per categorie mancanti
print("\nFlag speciali per categorie mancanti:")
for flag in ['FLAG_HE', 'FLAG_BRIDAL', 'FLAG_DIAMOND']:
    if flag in art_miss.columns:
        print(f"  {flag}: {art_miss[flag].value_counts(dropna=False).to_dict()}")

# 5E.4 — Join Articles × Transactions su ARTICLE_ID
art_ids_missing_cat = set(art_miss['ARTICLE_ID'].dropna().unique())
trs_article_ids = set(trs['ARTICLE_ID'].dropna().unique())
overlap = art_ids_missing_cat & trs_article_ids

print(f"\nARTICLE_ID nelle categorie mancanti: {len(art_ids_missing_cat):,}")
print(f"Overlap con ARTICLE_ID in Transactions: {len(overlap):,}")

if len(overlap) > 0:
    # Controlla con quale CATEG appaiono in Transactions
    trs_overlap = trs[trs['ARTICLE_ID'].isin(overlap)]
    print("\nCATEG usata in Transactions per articoli delle categorie 'mancanti':")
    print(trs_overlap['CATEG'].value_counts(dropna=False).to_string())

# Salva tabella
cat_summary = art.groupby('PRODUCT_CATEGORY').agg(
    n_articles=('ARTICLE_ID', 'count'),
    world_price_mean=('WORLD_PRICE', 'mean'),
    world_price_median=('WORLD_PRICE', 'median'),
    flag_he_pct=('FLAG_HE', lambda x: x.mean()*100 if x.dtype == bool or x.isin([0,1]).all() else np.nan),
).reset_index()
cat_summary['in_transactions'] = cat_summary['PRODUCT_CATEGORY'].isin(cats_present)
cat_summary.to_csv(OUT_TABLES / 'articles_category_analysis.csv', index=False)

print_finding(
    'CATEGORIE ARTICLES MANCANTI',
    f"Categorie mancanti: {sorted(cats_missing)}.\n"
    f"Articoli in categorie mancanti: {len(art_miss):,}.\n"
    f"Overlap ARTICLE_ID con Transactions: {len(overlap):,}.\n"
    "Ipotesi: le 5 categorie mancanti potrebbero essere \n"
    "  (a) linee non vendibili via canale clientela (HE/bridal/speciale), \n"
    "  (b) articoli storici dismessi, \n"
    "  (c) mapping CATEG Transactions != PRODUCT_CATEGORY Articles (solo 3 macro-categorie). \n"
    "Salvato: articles_category_analysis.csv"
)

---
## Fase 5F — CRC: copertura temporale e APPOINTMENT_DURATION

In [ ]:
print("=== 5F — CRC: COPERTURA TEMPORALE E APPOINTMENT_DURATION ===")

# Colonna duration (attenzione al typo nel dataset: APPOINTEMENT)
dur_col = [c for c in crc.columns if 'DURATION' in c.upper()]
print(f"Colonne duration trovate: {dur_col}")
dur_col = dur_col[0] if dur_col else None

# 5F.1 — Intervallo CREATION_DATE
print(f"\nCREATION_DATE min: {crc['CREATION_DATE'].min()}")
print(f"CREATION_DATE max: {crc['CREATION_DATE'].max()}")

# 5F.2 — CLIENT_ID unici in CRC
crc_clients = set(crc['CLIENT_ID'].dropna().unique())
print(f"\nCLIENT_ID unici in CRC: {len(crc_clients):,}")

# 5F.3 — Overlap con snapshot 2021
crc_in_2021 = crc_clients & clients_2021
print(f"CRC clients in snapshot 2021: {len(crc_in_2021):,} ({len(crc_in_2021)/len(clients_2021)*100:.1f}% dei clienti 2021)")

# 5F.4 — APPOINTMENT_DURATION missing per ORIGIN
if dur_col:
    print(f"\nMissing {dur_col} overall: {crc[dur_col].isna().mean()*100:.1f}%")
    print(f"\nMissing per ORIGIN:")
    dur_by_origin = crc.groupby('ORIGIN')[dur_col].agg(
        n_total='count',
        n_missing=lambda x: x.isna().sum(),
        pct_missing=lambda x: x.isna().mean()*100,
    ).reset_index()
    # Aggiunge n totale (inclusi missing)
    origin_totals = crc.groupby('ORIGIN').size().reset_index(name='n_totale')
    dur_by_origin = dur_by_origin.merge(origin_totals, on='ORIGIN')
    dur_by_origin['pct_missing_correct'] = (dur_by_origin['n_missing'] / dur_by_origin['n_totale'] * 100).round(1)
    dur_by_origin = dur_by_origin.sort_values('pct_missing_correct', ascending=False)
    print(dur_by_origin[['ORIGIN', 'n_totale', 'n_missing', 'pct_missing_correct']].to_string(index=False))
    dur_by_origin.to_csv(OUT_TABLES / 'crc_appointment_duration_by_origin.csv', index=False)
    
    # Valutazione MAR vs MNAR
    pct_values = dur_by_origin['pct_missing_correct'].values
    varianza = np.std(pct_values)
    print(f"\nStd % missing per ORIGIN: {varianza:.1f}%")
    if varianza > 10:
        diagnosi = "MAR — il missing è concentrato su certi ORIGIN (correlato a canale)"
    else:
        diagnosi = "MNAR — il missing è distribuito uniformemente indipendentemente dall'ORIGIN"
    print(f"Diagnosi: {diagnosi}")
else:
    diagnosi = "Colonna duration non trovata"

# Overlap di CRC con lo snapshot 2021 (solo post DATE_TARGET 2021)
snap_2021 = pd.Timestamp('2021-01-01')
crc_post2021 = crc[crc['CREATION_DATE'] > snap_2021]
crc_pre2021  = crc[crc['CREATION_DATE'] <= snap_2021]
print(f"\nAppuntamenti CRC <= 2021: {len(crc_pre2021):,}")
print(f"Appuntamenti CRC >  2021: {len(crc_post2021):,}")
print(f"CLIENT_ID unici CRC <= 2021: {crc_pre2021['CLIENT_ID'].nunique():,}")

print_finding(
    'CRC COPERTURA E APPOINTMENT_DURATION',
    f"CRC copre {crc['CREATION_DATE'].min()} — {crc['CREATION_DATE'].max()}.\n"
    f"CLIENT_ID unici: {len(crc_clients):,}, di cui {len(crc_in_2021):,} nello snapshot 2021 ({len(crc_in_2021)/len(clients_2021)*100:.1f}%).\n"
    f"APPOINTMENT_DURATION missing: {crc[dur_col].isna().mean()*100:.1f}% overall.\n"
    f"Diagnosi missing: {diagnosi}.\n"
    "Salvato: crc_appointment_duration_by_origin.csv"
)

---
## Fase 5G — CCP: copertura temporale e consistenza date

In [ ]:
print("=== 5G — CCP: COPERTURA TEMPORALE E DATE CONSISTENCY ===")

# 5G.1 — Range date
print(f"CREATION_DATE min: {ccp['CREATION_DATE'].min()}")
print(f"CREATION_DATE max: {ccp['CREATION_DATE'].max()}")
print(f"SALE_DATE min:     {ccp['SALE_DATE'].min()}")
print(f"SALE_DATE max:     {ccp['SALE_DATE'].max()}")

# 5G.2 — CLIENT_ID unici
ccp_clients = set(ccp['CLIENT_ID'].dropna().unique())
print(f"\nCLIENT_ID unici in CCP: {len(ccp_clients):,}")

# 5G.3 — Overlap con snapshot 2021
ccp_in_2021 = ccp_clients & clients_2021
print(f"CCP clients in snapshot 2021: {len(ccp_in_2021):,} ({len(ccp_in_2021)/len(clients_2021)*100:.1f}% dei clienti 2021)")

# 5G.4 — SALE_DATE <= CREATION_DATE
valid_mask = ccp['SALE_DATE'].notna() & ccp['CREATION_DATE'].notna()
violations = (ccp.loc[valid_mask, 'SALE_DATE'] > ccp.loc[valid_mask, 'CREATION_DATE']).sum()
print(f"\nViolazioni SALE_DATE > CREATION_DATE: {violations:,} ({violations/valid_mask.sum()*100:.2f}% delle righe valide)")

# Distribuzione gap CREATION - SALE
if valid_mask.sum() > 0:
    gap_days = (ccp.loc[valid_mask, 'CREATION_DATE'] - ccp.loc[valid_mask, 'SALE_DATE']).dt.days
    print("\nGap CREATION_DATE - SALE_DATE (giorni):")
    print(gap_days.describe().to_string())
    print(f"Gap negativo (sale > creation): {(gap_days < 0).sum():,}")
    print(f"Gap = 0 (stessa data): {(gap_days == 0).sum():,}")
    print(f"Gap > 365 giorni: {(gap_days > 365).sum():,}")

# 5G.5 — % missing SALE_DATE per FLAG_GIFT
print("\n% missing SALE_DATE per FLAG_GIFT:")
sale_miss = ccp.groupby('FLAG_GIFT')['SALE_DATE'].agg(
    n_total='count',
    n_missing=lambda x: x.isna().sum(),
    pct_missing=lambda x: x.isna().mean()*100
)
# Conta totale (compreso missing)
total_by_gift = ccp.groupby('FLAG_GIFT').size().rename('n_totale')
sale_miss_df = sale_miss.join(total_by_gift)
sale_miss_df['pct_missing'] = (sale_miss_df['n_missing'] / sale_miss_df['n_totale'] * 100).round(2)
print(sale_miss_df.to_string())

# Controllo overlap CCP pre/post 2021
ccp_pre2021  = ccp[ccp['CREATION_DATE'] <= snap_2021]
ccp_post2021 = ccp[ccp['CREATION_DATE'] > snap_2021]
print(f"\nCCP registrazioni <= 2021: {len(ccp_pre2021):,}")
print(f"CCP registrazioni >  2021: {len(ccp_post2021):,}")
print(f"CLIENT_ID unici CCP <= 2021: {ccp_pre2021['CLIENT_ID'].nunique():,}")

# Salva tabella
ccp_summary = pd.DataFrame([{
    'N righe': len(ccp),
    'N CLIENT_ID unici': len(ccp_clients),
    'N clienti in snapshot 2021': len(ccp_in_2021),
    '% clienti 2021 in CCP': round(len(ccp_in_2021)/len(clients_2021)*100, 1),
    'CREATION_DATE min': str(ccp['CREATION_DATE'].min()),
    'CREATION_DATE max': str(ccp['CREATION_DATE'].max()),
    'SALE_DATE min': str(ccp['SALE_DATE'].min()),
    'SALE_DATE max': str(ccp['SALE_DATE'].max()),
    'N violazioni SALE > CREATION': int(violations),
}])
ccp_summary.to_csv(OUT_TABLES / 'quality_ccp.csv', index=False)

print_finding(
    'CCP COPERTURA E DATE CONSISTENCY',
    f"CCP: {len(ccp):,} righe, {len(ccp_clients):,} CLIENT_ID unici.\n"
    f"Coverage snapshot 2021: {len(ccp_in_2021):,} clienti ({len(ccp_in_2021)/len(clients_2021)*100:.1f}%).\n"
    f"Violazioni SALE_DATE > CREATION_DATE: {violations:,}.\n"
    "Salvato: quality_ccp.csv"
)

---
## Fase 6A — Matrice di coverage referential integrity

In [ ]:
print("=== 6A — MATRICE REFERENTIAL INTEGRITY ===")

# Insiemi CLIENT_ID per dataset
id_sets = {
    'Aggregated_2021': clients_2021,
    'Aggregated_ALL':  set(agg['CLIENT_ID'].dropna().unique()),
    'Transactions':    set(trs['CLIENT_ID'].dropna().unique()),
    'Clients':         set(cli['CLIENT_ID'].dropna().unique()),
    'CRC':             crc_clients,
    'CCP':             ccp_clients,
}

for name, s in id_sets.items():
    print(f"  {name}: {len(s):,} CLIENT_ID unici")

# Calcola matrice di overlap
dataset_names = list(id_sets.keys())
matrix_rows = []

for i, name_a in enumerate(dataset_names):
    for j, name_b in enumerate(dataset_names):
        if i >= j:
            continue
        set_a = id_sets[name_a]
        set_b = id_sets[name_b]
        intersection = set_a & set_b
        only_a = set_a - set_b
        only_b = set_b - set_a
        matrix_rows.append({
            'Dataset_A': name_a,
            'Dataset_B': name_b,
            'N_A': len(set_a),
            'N_B': len(set_b),
            'N_intersection': len(intersection),
            'N_only_A': len(only_a),
            'N_only_B': len(only_b),
            'pct_A_in_B': round(len(intersection)/len(set_a)*100, 1) if len(set_a) > 0 else 0,
            'pct_B_in_A': round(len(intersection)/len(set_b)*100, 1) if len(set_b) > 0 else 0,
        })

ri_df = pd.DataFrame(matrix_rows)
print("\nMatrice di coverage (pct_A_in_B = % di A trovata in B):")
print(ri_df[['Dataset_A', 'Dataset_B', 'N_intersection', 'N_only_A', 'N_only_B', 'pct_A_in_B', 'pct_B_in_A']].to_string(index=False))

ri_df.to_csv(OUT_TABLES / 'referential_integrity_matrix.csv', index=False)
print("\nSalvato: referential_integrity_matrix.csv")

---
## Fase 6B — Clienti orfani: analisi caratteristiche

In [ ]:
print("=== 6B — CLIENTI ORFANI ===")

# 6B.1 — In Transactions ma NON in Aggregated_2021
trs_not_agg2021 = id_sets['Transactions'] - clients_2021
print(f"\nIn Transactions ma NON in Aggregated_2021: {len(trs_not_agg2021):,}")
if len(trs_not_agg2021) > 0:
    trs_orphan = trs[trs['CLIENT_ID'].isin(trs_not_agg2021)]
    print(f"  Transazioni di questi clienti: {len(trs_orphan):,}")
    print(f"  Anno TRS_DATE (distribuzione):")
    print(trs_orphan['TRS_DATE'].dt.year.value_counts().sort_index().to_string())
    # Verifica se sono in altri snapshot di Aggregated
    in_other_snap = id_sets['Transactions'] & id_sets['Aggregated_ALL'] - clients_2021
    print(f"  Di questi, presenti in altri snapshot Aggregated: {len(in_other_snap):,}")
    only_trs = trs_not_agg2021 - id_sets['Aggregated_ALL']
    print(f"  NON in nessuno snapshot Aggregated: {len(only_trs):,}")

# 6B.2 — In Aggregated_2021 ma NON in Transactions
agg2021_not_trs = clients_2021 - id_sets['Transactions']
print(f"\nIn Aggregated_2021 ma NON in Transactions: {len(agg2021_not_trs):,}")
if len(agg2021_not_trs) > 0:
    agg_orphan = agg_2021[agg_2021['CLIENT_ID'].isin(agg2021_not_trs)]
    if 'TO_FULL_HIST' in agg_orphan.columns:
        print(f"  TO_FULL_HIST > 0 (nonostante assenza in Transactions): {(agg_orphan['TO_FULL_HIST'] > 0).sum():,}")
        print(f"  TO_FULL_HIST = 0: {(agg_orphan['TO_FULL_HIST'] == 0).sum():,}")
        print(f"  TO_FULL_HIST NULL: {agg_orphan['TO_FULL_HIST'].isna().sum():,}")
    if 'NB_TRS_HIST' in agg_orphan.columns:
        print(f"  NB_TRS_HIST > 0: {(agg_orphan['NB_TRS_HIST'] > 0).sum():,}")

# 6B.3 — In CRC ma NON in Aggregated_2021
crc_not_agg2021 = crc_clients - clients_2021
print(f"\nIn CRC ma NON in Aggregated_2021: {len(crc_not_agg2021):,}")
if len(crc_not_agg2021) > 0:
    crc_orphan = crc[crc['CLIENT_ID'].isin(crc_not_agg2021)]
    print(f"  CREATION_DATE min/max: {crc_orphan['CREATION_DATE'].min()} — {crc_orphan['CREATION_DATE'].max()}")
    print(f"  Potrebbero essere clienti acquisiti dopo 2021.")

# 6B.4 — In CCP ma NON in Aggregated_2021
ccp_not_agg2021 = ccp_clients - clients_2021
print(f"\nIn CCP ma NON in Aggregated_2021: {len(ccp_not_agg2021):,}")
if len(ccp_not_agg2021) > 0:
    ccp_orphan = ccp[ccp['CLIENT_ID'].isin(ccp_not_agg2021)]
    print(f"  CREATION_DATE min/max: {ccp_orphan['CREATION_DATE'].min()} — {ccp_orphan['CREATION_DATE'].max()}")

print_finding(
    'CLIENTI ORFANI',
    f"TRS not in Agg_2021: {len(trs_not_agg2021):,}.\n"
    f"Agg_2021 not in TRS: {len(agg2021_not_trs):,}.\n"
    f"CRC not in Agg_2021: {len(crc_not_agg2021):,}.\n"
    f"CCP not in Agg_2021: {len(ccp_not_agg2021):,}.\n"
)

---
## Fase 6C — ARTICLE_ID: coverage tra Articles e Transactions

In [ ]:
print("=== 6C — ARTICLE_ID COVERAGE ===")

art_ids_in_art = set(art['ARTICLE_ID'].dropna().unique())
art_ids_in_trs = set(trs['ARTICLE_ID'].dropna().unique())

trs_in_art     = art_ids_in_trs & art_ids_in_art
trs_not_in_art = art_ids_in_trs - art_ids_in_art
art_not_in_trs = art_ids_in_art - art_ids_in_trs

print(f"ARTICLE_ID unici in Transactions:    {len(art_ids_in_trs):,}")
print(f"ARTICLE_ID unici in Articles:        {len(art_ids_in_art):,}")
print(f"Match (TRS ∩ Articles):              {len(trs_in_art):,} ({len(trs_in_art)/len(art_ids_in_trs)*100:.1f}% di TRS)")
print(f"In TRS ma NON in Articles (orfani):  {len(trs_not_in_art):,} ({len(trs_not_in_art)/len(art_ids_in_trs)*100:.1f}% di TRS)")
print(f"In Articles ma NON in TRS:           {len(art_not_in_trs):,} ({len(art_not_in_trs)/len(art_ids_in_art)*100:.1f}% di Articles)")

# Caratteristiche articoli orfani (in TRS ma non in Articles)
if len(trs_not_in_art) > 0:
    trs_orphan_arts = trs[trs['ARTICLE_ID'].isin(trs_not_in_art)]
    print(f"\nTransazioni con ARTICLE_ID orfano (non in Articles): {len(trs_orphan_arts):,}")
    print("  CHANNEL:")
    print(trs_orphan_arts['CHANNEL'].value_counts(dropna=False).to_string())
    print("  CATEG:")
    print(trs_orphan_arts['CATEG'].value_counts(dropna=False).to_string())
    print("  Anno TRS_DATE:")
    print(trs_orphan_arts['TRS_DATE'].dt.year.value_counts().sort_index().to_string())
    print("  TRS_CATEG:")
    print(trs_orphan_arts['TRS_CATEG'].value_counts(dropna=False).to_string())

art_coverage = pd.DataFrame([{
    'ARTICLE_ID in Transactions': len(art_ids_in_trs),
    'ARTICLE_ID in Articles': len(art_ids_in_art),
    'Match TRS in Art': len(trs_in_art),
    'pct TRS in Art': round(len(trs_in_art)/len(art_ids_in_trs)*100, 1),
    'Orfani TRS not in Art': len(trs_not_in_art),
    'Art not in TRS': len(art_not_in_trs),
}])
art_coverage.to_csv(OUT_TABLES / 'article_id_coverage.csv', index=False)

print_finding(
    'ARTICLE_ID COVERAGE',
    f"TRS ARTICLE_ID unici: {len(art_ids_in_trs):,}.\n"
    f"Trovati in Articles: {len(trs_in_art):,} ({len(trs_in_art)/len(art_ids_in_trs)*100:.1f}%).\n"
    f"Orfani (TRS senza anagrafica Articles): {len(trs_not_in_art):,}.\n"
    "Salvato: article_id_coverage.csv"
)

---
## Fase 6D — Sanity check numerico finale

In [ ]:
print("=== 6D — SANITY CHECK AGGREGATI ===")

# 6D.1 — Somma TO_WITHOUTTAX_EUR_CONST in TRS vs TO_FULL_HIST in Aggregated
# Filtro: solo clienti dello snapshot 2021 e transazioni <= 2021
trs_2021_clients = trs[
    (trs['CLIENT_ID'].isin(clients_2021)) &
    (trs['TRS_DATE'] <= snap_2021)
]

sum_to_trs   = trs_2021_clients[col_spend].sum()
sum_to_agg   = agg_2021['TO_FULL_HIST'].sum() if 'TO_FULL_HIST' in agg_2021.columns else np.nan

print(f"Somma TO_WITHOUTTAX_EUR_CONST in TRS (clienti 2021, date <= 2021): {sum_to_trs:,.0f} EUR")
print(f"Somma TO_FULL_HIST in Aggregated_2021:                              {sum_to_agg:,.0f} EUR")
if sum_to_agg and sum_to_trs:
    ratio = sum_to_trs / sum_to_agg
    print(f"Rapporto TRS / AGG: {ratio:.3f}")
    if 0.5 <= ratio <= 2.0:
        print("  => Stesso ordine di grandezza — COERENTE")
    else:
        print("  => Discrepanza significativa — da investigare")

# 6D.2 — N. medio transazioni per cliente
mean_trs_per_client_raw = trs_2021_clients.groupby('CLIENT_ID').size().mean()
print(f"\nN. medio transazioni per cliente (TRS, clienti 2021, pre-2021): {mean_trs_per_client_raw:.1f}")

# Cerca colonna NB_TRS in Aggregated
nb_cols = [c for c in agg_2021.columns if c.startswith('NB_TRS')]
print(f"Colonne NB_TRS in Aggregated_2021: {nb_cols}")
if nb_cols:
    for col in nb_cols[:3]:  # Primi 3
        mean_nb = agg_2021[col].mean()
        print(f"  {col} — media: {mean_nb:.1f}, mediana: {agg_2021[col].median():.1f}")

# Spesa media per cliente
mean_spend_trs = trs_2021_clients.groupby('CLIENT_ID')[col_spend].sum().mean()
mean_spend_agg = agg_2021['TO_FULL_HIST'].mean() if 'TO_FULL_HIST' in agg_2021.columns else np.nan
print(f"\nSpesa media per cliente in TRS: {mean_spend_trs:,.0f} EUR")
print(f"Spesa media per cliente in AGG (TO_FULL_HIST): {mean_spend_agg:,.0f} EUR")

sanity_df = pd.DataFrame([{
    'Sum TO_TRS (clienti 2021, pre-2021)': round(sum_to_trs, 0),
    'Sum TO_FULL_HIST (Agg 2021)': round(sum_to_agg, 0) if sum_to_agg else None,
    'Ratio TRS/AGG': round(ratio, 3) if (sum_to_agg and sum_to_trs) else None,
    'Mean TRS per cliente (TRS)': round(mean_trs_per_client_raw, 1),
    'Mean spend per cliente (TRS)': round(mean_spend_trs, 0),
    'Mean spend per cliente (AGG)': round(mean_spend_agg, 0) if mean_spend_agg else None,
}])
sanity_df.to_csv(OUT_TABLES / 'sanity_check_aggregates.csv', index=False)

print_finding(
    'SANITY CHECK AGGREGATI',
    f"Somma TRS: {sum_to_trs:,.0f} EUR vs Aggregated TO_FULL_HIST: {sum_to_agg:,.0f} EUR.\n"
    f"Rapporto: {ratio:.3f} — {'COERENTE' if 0.5 <= ratio <= 2.0 else 'DISCREPANZA'}.\n"
    "N.b.: differenza attesa per: transazioni escluse per data, resi, canali diversi.\n"
    "Salvato: sanity_check_aggregates.csv"
)

---
## Riepilogo finale — quality_transactions.csv e quality_crc_ccp.csv

In [ ]:
# Consolida quality_transactions.csv
qt = pd.DataFrame([
    {'Check': 'TRS_DATE range', 'Value': f"{trs['TRS_DATE'].min()} — {trs['TRS_DATE'].max()}"},
    {'Check': 'N transazioni totali', 'Value': len(trs)},
    {'Check': 'Transazioni post-snapshot-2021', 'Value': int(len(trs_leak_last))},
    {'Check': '% post-2021', 'Value': f"{len(trs_leak_last)/len(trs)*100:.1f}%"},
    {'Check': 'SERIAL_NUMBER NULL', 'Value': n_null},
    {'Check': '% SERIAL_NUMBER NULL', 'Value': f"{n_null/len(trs)*100:.1f}%"},
    {'Check': 'SERIAL_NUMBER placeholder', 'Value': PLACEHOLDER},
    {'Check': 'N placeholder occorrenze', 'Value': n_ph},
    {'Check': f'{col_spend} < 0', 'Value': int(neg_mask.sum())},
    {'Check': f'% {col_spend} < 0', 'Value': f"{neg_mask.mean()*100:.3f}%"},
    {'Check': f'{col_spend} NULL', 'Value': int(null_mask.sum())},
    {'Check': 'ARTICLE_WWPRICE = 0', 'Value': int(zero_price_mask.sum())},
    {'Check': 'Categorie mancanti Articles', 'Value': str(sorted(cats_missing))},
    {'Check': 'Articoli in categorie mancanti', 'Value': len(art_miss)},
])
qt.to_csv(OUT_TABLES / 'quality_transactions.csv', index=False)

# Consolida quality_crc_ccp.csv
qcc = pd.DataFrame([
    {'Dataset': 'CRC', 'Check': 'CREATION_DATE range', 'Value': f"{crc['CREATION_DATE'].min()} — {crc['CREATION_DATE'].max()}"},
    {'Dataset': 'CRC', 'Check': 'CLIENT_ID unici', 'Value': len(crc_clients)},
    {'Dataset': 'CRC', 'Check': 'CLIENT_ID in snapshot 2021', 'Value': len(crc_in_2021)},
    {'Dataset': 'CRC', 'Check': '% clienti 2021 coperti', 'Value': f"{len(crc_in_2021)/len(clients_2021)*100:.1f}%"},
    {'Dataset': 'CRC', 'Check': 'APPOINTMENT_DURATION missing %', 'Value': f"{crc[dur_col].isna().mean()*100:.1f}%" if dur_col else 'N/A'},
    {'Dataset': 'CRC', 'Check': 'Diagnosi missing duration', 'Value': diagnosi},
    {'Dataset': 'CCP', 'Check': 'CREATION_DATE range', 'Value': f"{ccp['CREATION_DATE'].min()} — {ccp['CREATION_DATE'].max()}"},
    {'Dataset': 'CCP', 'Check': 'CLIENT_ID unici', 'Value': len(ccp_clients)},
    {'Dataset': 'CCP', 'Check': 'CLIENT_ID in snapshot 2021', 'Value': len(ccp_in_2021)},
    {'Dataset': 'CCP', 'Check': '% clienti 2021 coperti', 'Value': f"{len(ccp_in_2021)/len(clients_2021)*100:.1f}%"},
    {'Dataset': 'CCP', 'Check': 'N violazioni SALE_DATE > CREATION_DATE', 'Value': int(violations)},
])
qcc.to_csv(OUT_TABLES / 'quality_crc_ccp.csv', index=False)

print("Salvati:")
print("  output/tables/quality_transactions.csv")
print("  output/tables/quality_crc_ccp.csv")
print("  output/tables/referential_integrity_matrix.csv")
print("  output/tables/serial_number_analysis.csv")
print("  output/tables/quality_transactions_negatives.csv")
print("  output/tables/articles_category_analysis.csv")
print("  output/tables/crc_appointment_duration_by_origin.csv")
print("  output/tables/quality_ccp.csv")
print("  output/tables/article_id_coverage.csv")
print("  output/tables/sanity_check_aggregates.csv")
print("  output/tables/transactions_temporal_coverage.csv")

print("\n=== NOTEBOOK COMPLETATO ===")